# 09 - Final Evaluation, Horizon Degradation, and Scenario Analysis

Consolidate all evidence into portfolio-grade conclusions.


## Forecast Horizon Degradation

**Definition**  
As horizon increases, uncertainty compounds and signal decays.

**Theory**  
Error typically increases sublinearly or superlinearly depending on regime volatility.

**Mathematical Intuition**  
For horizon `h`, expected forecast variance generally increases with `h` under diffusion-like dynamics.

**Financial Intuition**  
Short horizons capture microstructure and momentum; long horizons are dominated by macro/exogenous noise.

**Business Impact**  
Horizon-aware deployment avoids overpromising accuracy.

**Real-World Example**  
A 1-day model may be useful for tactical positioning while 30-day output supports scenario planning only.

**Visual Explanation**  
Code cells below generate real plots from Apple market data.

**Code Explanation**  
Each code block is annotated inline and uses shared production modules from `src/`.

**Interpretation of Results**  
After each output, interpret what signal quality, risk, and forecasting reliability imply.


In [ ]:

import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.forecast_pipeline import ForecastingFramework
from src.data_loader import load_stock_data, split_data
from src.features import create_features
from src.evaluation import regression_metrics
from src.visualization import *

OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
FAST_NOTEBOOK_MODE = os.getenv('NOTEBOOK_FULL_RUN', '0') != '1'

def make_framework():
    framework = ForecastingFramework(str(CONFIG_PATH))
    if FAST_NOTEBOOK_MODE:
        framework.config['models']['automl']['lazypredict'] = False
        framework.config['models']['automl']['pycaret'] = False
        framework.config['models']['automl']['flaml'] = False
        framework.config['models']['deep_learning']['epochs'] = 8
        framework.config['models']['deep_learning']['early_stopping_patience'] = 3
        framework.config['weight_optimization']['methods'] = ['grid']
    return framework

framework = make_framework()


In [ ]:

framework = make_framework()
framework.load_data()

summary_rows = []
for h in ([1] if FAST_NOTEBOOK_MODE else framework.config['features']['horizons']):
    out = framework.run_horizon(h)
    best_hybrid = out['hybrid']['leaderboard'].iloc[0]
    summary_rows.append({
        'horizon': h,
        'best_hybrid_model': best_hybrid['model'],
        'best_hybrid_rmse': best_hybrid['rmse'],
        'best_hybrid_mape': best_hybrid['mape'],
    })

summary_df = pd.DataFrame(summary_rows).sort_values('horizon')
display(summary_df)
summary_df.to_csv(OUT / 'tables/09_final_horizon_summary.csv', index=False)


In [ ]:

# Scenario analysis on horizon=1 using baseline features and best baseline model
from src.models import MODEL_REGISTRY

ds = framework.prepare_horizon_dataset(1)
model = MODEL_REGISTRY['Random Forest']
model.fit(ds.X_train, ds.y_train)

base_pred = model.predict(ds.X_test)
base_metrics = regression_metrics(ds.y_test, base_pred)

# Scenario 1: volume spike
X_volume_spike = ds.X_test.copy()
X_volume_spike[:, ds.feature_columns.index('Volume')] *= 1.5 if 'Volume' in ds.feature_columns else 1.0
pred_volume_spike = model.predict(X_volume_spike)

# Scenario 2: volatility increase (perturb high/low-related engineered features)
X_vol_shock = ds.X_test.copy()
for name in ds.feature_columns:
    if 'rolling_std' in name or 'atr' in name or 'bb_width' in name:
        idx = ds.feature_columns.index(name)
        X_vol_shock[:, idx] *= 1.25
pred_vol_shock = model.predict(X_vol_shock)

# Scenario 3: trend reversal
X_trend_reverse = ds.X_test.copy()
for name in ds.feature_columns:
    if 'momentum' in name or 'roc' in name:
        idx = ds.feature_columns.index(name)
        X_trend_reverse[:, idx] *= -1
pred_trend_reverse = model.predict(X_trend_reverse)

scenario_df = pd.DataFrame([
    {'scenario': 'base', 'rmse': regression_metrics(ds.y_test, base_pred)['rmse']},
    {'scenario': 'volume_spike', 'rmse': regression_metrics(ds.y_test, pred_volume_spike)['rmse']},
    {'scenario': 'volatility_shock', 'rmse': regression_metrics(ds.y_test, pred_vol_shock)['rmse']},
    {'scenario': 'trend_reversal', 'rmse': regression_metrics(ds.y_test, pred_trend_reverse)['rmse']},
])

display(scenario_df)
scenario_df.to_csv(OUT / 'tables/09_scenario_analysis.csv', index=False)


Set `NOTEBOOK_FULL_RUN=1` before execution to run all configured horizons in this notebook.


## Final Lessons Learned

- Hybrid learning improves robustness over isolated model families.
- Weight learning outperforms fixed blending under regime changes.
- Backtesting strategy materially changes perceived model quality.
- Explainability is necessary for trustworthy financial deployment.
- Horizon should drive model and risk policy, not only metric ranking.
